### imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.fft as fft

### helper functions

In [ ]:

# ────────────────────────────────────────────────
#  Training setup — the modern minimal way
# ────────────────────────────────────────────────

def get_dataloaders(batch_size=128):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                              shuffle=True, num_workers=2, pin_memory=True)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                             shuffle=False, num_workers=2, pin_memory=True)

    return trainloader, testloader


# Quick training loop
def train(model, epochs=40):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # helps a bit

    trainloader, testloader = get_dataloaders()

    print("Starting training...")
    for epoch in range(epochs):  # 60–90 epochs is usually enough
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            if i % 100 == 99:
                print(f"[{epoch+1}, {i+1:5d}] loss: {running_loss/100:.3f} | acc: {100.*correct/total:.2f}%")
                running_loss = 0.0
                correct = total = 0

        # Quick test accuracy
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        acc = 100. * correct / total
        print(f"Epoch {epoch+1:2d}  ─  Test accuracy: {acc:.2f}%")

        scheduler.step()

    print("Done.")
    return model

### basic architecture

In [ ]:
# ────────────────────────────────────────────────
#  Most basic solid CNN for CIFAR-10 (~78–82% possible)
# ────────────────────────────────────────────────

class TinyConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3,  32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),                                     # 32×32 → 16×16

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.Conv2d(128,128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),                                     # 16×16 → 8×8

            # Block 3 — very light
            nn.Conv2d(128,256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            nn.AdaptiveAvgPool2d(1)                                 # → 1×1
        )

        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)   # or x = x.flatten(1)
        x = self.classifier(x)
        return x

In [ ]:
model = TinyConvNet()
train(model, epochs=40)
torch.save(model.state_dict(), "./data/basic-architecture-model-40-epochs")

100%|██████████| 170M/170M [00:03<00:00, 42.8MB/s]


Starting training...
[1,   100] loss: 1.768 | acc: 40.77%
[1,   200] loss: 1.496 | acc: 54.93%
[1,   300] loss: 1.382 | acc: 60.88%
Epoch  1  ─  Test accuracy: 64.63%
[2,   100] loss: 1.223 | acc: 68.63%
[2,   200] loss: 1.169 | acc: 71.72%
[2,   300] loss: 1.126 | acc: 73.84%
Epoch  2  ─  Test accuracy: 72.29%
[3,   100] loss: 1.044 | acc: 77.88%
[3,   200] loss: 1.035 | acc: 78.47%
[3,   300] loss: 1.027 | acc: 78.86%
Epoch  3  ─  Test accuracy: 78.43%
[4,   100] loss: 0.957 | acc: 82.27%
[4,   200] loss: 0.968 | acc: 81.81%
[4,   300] loss: 0.953 | acc: 82.43%
Epoch  4  ─  Test accuracy: 82.23%
[5,   100] loss: 0.912 | acc: 84.39%
[5,   200] loss: 0.910 | acc: 84.62%
[5,   300] loss: 0.903 | acc: 84.74%
Epoch  5  ─  Test accuracy: 83.17%
[6,   100] loss: 0.866 | acc: 86.66%
[6,   200] loss: 0.877 | acc: 86.05%
[6,   300] loss: 0.867 | acc: 86.58%
Epoch  6  ─  Test accuracy: 84.40%
[7,   100] loss: 0.843 | acc: 87.95%
[7,   200] loss: 0.845 | acc: 87.84%
[7,   300] loss: 0.847 | acc:

### model with frelu

In [ ]:
def freq_mag_softshrink(x, lam=1e-2, dims=(-2, -1), eps=1e-8):
    z = torch.fft.fftn(x, dim=dims)
    mag = torch.abs(z)
    scale = torch.clamp(mag - lam, min=0.0) / (mag + eps)
    z2 = z * scale
    y = torch.fft.ifftn(z2, dim=dims)
    return y.real

class fReLU(nn.Module):
    def __init__(self, lam=1e-2, dims=(-2, -1), learnable=True):
        super().__init__()
        self.dims = dims
        if learnable:
            self.lam = nn.Parameter(torch.tensor(float(lam)))
        else:
            self.register_buffer("lam", torch.tensor(float(lam)))

    def forward(self, x):
        lam = torch.relu(self.lam)
        return freq_mag_softshrink(x, lam=lam, dims=self.dims)

In [ ]:
class FreluModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3,  32, 3, padding=1), fReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1), fReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),                                     # 32×32 → 16×16

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1), fReLU(), nn.BatchNorm2d(128),
            nn.Conv2d(128,128, 3, padding=1), fReLU(), nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),                                     # 16×16 → 8×8

            # Block 3 — very light
            nn.Conv2d(128,256, 3, padding=1), fReLU(), nn.BatchNorm2d(256),
            nn.AdaptiveAvgPool2d(1)                                 # → 1×1
        )

        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)   # or x = x.flatten(1)
        x = self.classifier(x)
        return x

In [ ]:
model = FreluModel()
train(model, epochs=40)
torch.save(model.state_dict(), "./data/basic-architecture-with-frelu-model-40-epochs")

Starting training...
[1,   100] loss: 1.952 | acc: 31.52%
[1,   200] loss: 1.815 | acc: 38.66%
[1,   300] loss: 1.716 | acc: 44.21%
Epoch  1  ─  Test accuracy: 49.30%
[2,   100] loss: 1.626 | acc: 49.89%
[2,   200] loss: 1.598 | acc: 51.55%
[2,   300] loss: 1.566 | acc: 52.53%
Epoch  2  ─  Test accuracy: 54.11%
[3,   100] loss: 1.519 | acc: 55.61%
[3,   200] loss: 1.515 | acc: 55.86%
[3,   300] loss: 1.494 | acc: 56.46%
Epoch  3  ─  Test accuracy: 56.28%
[4,   100] loss: 1.474 | acc: 57.59%
[4,   200] loss: 1.466 | acc: 58.44%
[4,   300] loss: 1.448 | acc: 58.97%
Epoch  4  ─  Test accuracy: 54.73%
[5,   100] loss: 1.439 | acc: 59.65%
[5,   200] loss: 1.414 | acc: 61.57%
[5,   300] loss: 1.411 | acc: 60.95%
Epoch  5  ─  Test accuracy: 61.80%
[6,   100] loss: 1.389 | acc: 62.16%
[6,   200] loss: 1.396 | acc: 62.36%
[6,   300] loss: 1.382 | acc: 62.86%
Epoch  6  ─  Test accuracy: 61.91%
[7,   100] loss: 1.366 | acc: 63.82%
[7,   200] loss: 1.365 | acc: 63.27%
[7,   300] loss: 1.366 | acc:

### fReLU but different

In [ ]:
class fReLU(nn.Module):
    def __init__(self, channels, dims=(-2,-1), init_lam=0.5):
        super().__init__()
        self.dims = dims
        # one lambda per output channel
        self.lam = nn.Parameter(torch.full((channels,), float(init_lam)))

    def forward(self, x):
        # x : [B, C, H, W]
        lam = torch.relu(self.lam)[:, None, None, None]   # [C,1,1,1] → broadcast
        z = torch.fft.fftn(x, dim=self.dims)
        mag = torch.abs(z)
        scale = torch.clamp(mag - lam, min=0.0) / (mag + 1e-8)
        z2 = z * scale
        y = torch.fft.ifftn(z2, dim=self.dims).real
        return y

In [ ]:
class FreluModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3,  32, 3, padding=1), fReLU(32), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1), fReLU(64), nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),                                     # 32×32 → 16×16

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1), fReLU(128), nn.BatchNorm2d(128),
            nn.Conv2d(128,128, 3, padding=1), fReLU(128), nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),                                     # 16×16 → 8×8

            # Block 3 — very light
            nn.Conv2d(128,256, 3, padding=1), fReLU(256), nn.BatchNorm2d(256),
            nn.AdaptiveAvgPool2d(1)                                 # → 1×1
        )

        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)   # or x = x.flatten(1)
        x = self.classifier(x)
        return x

### basic architecture with residual blocks

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()  # or your fReLU
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Projection if channels change
        self.proj = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        # Amp on identity (skip)
        if self.proj is not None:
            identity = self.proj(identity)

        out = out + identity
        out = self.relu(out)  # post-add activation
        return out

class MiniResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            FreqAmpResBlock(3, 64),
            nn.MaxPool2d(2),
            FreqAmpResBlock(64, 128),
            nn.MaxPool2d(2),
            FreqAmpResBlock(128, 256),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

In [ ]:
model = MiniResNet()
train(model, epochs=40)
torch.save(model.state_dict(), "./data/basic-architecture-with-res-blocks-40-epochs")

Starting training...
[1,   100] loss: 1.760 | acc: 40.43%
[1,   200] loss: 1.478 | acc: 55.04%
[1,   300] loss: 1.360 | acc: 61.44%
Epoch  1  ─  Test accuracy: 53.26%
[2,   100] loss: 1.200 | acc: 69.48%
[2,   200] loss: 1.168 | acc: 71.84%
[2,   300] loss: 1.120 | acc: 73.11%
Epoch  2  ─  Test accuracy: 74.39%
[3,   100] loss: 1.055 | acc: 76.66%
[3,   200] loss: 1.009 | acc: 79.32%
[3,   300] loss: 1.002 | acc: 79.02%
Epoch  3  ─  Test accuracy: 79.37%
[4,   100] loss: 0.961 | acc: 80.96%
[4,   200] loss: 0.955 | acc: 82.09%
[4,   300] loss: 0.936 | acc: 82.72%
Epoch  4  ─  Test accuracy: 81.57%
[5,   100] loss: 0.899 | acc: 84.48%
[5,   200] loss: 0.903 | acc: 84.32%
[5,   300] loss: 0.898 | acc: 84.55%
Epoch  5  ─  Test accuracy: 82.79%
[6,   100] loss: 0.859 | acc: 86.20%
[6,   200] loss: 0.861 | acc: 86.21%
[6,   300] loss: 0.859 | acc: 86.16%
Epoch  6  ─  Test accuracy: 82.69%
[7,   100] loss: 0.823 | acc: 88.09%
[7,   200] loss: 0.838 | acc: 87.21%
[7,   300] loss: 0.828 | acc:

### residuals with frequency based feature amplification

In [ ]:
class FreqAmp(nn.Module):
    """Amplifies underused freq components (low-magnitude ones)."""
    def __init__(self, amp_factor=2.0, thresh_quantile=0.3, dims=(-2, -1)):
        super().__init__()
        self.amp_factor = amp_factor  # how much to boost weak freq
        self.thresh_quantile = thresh_quantile  # e.g., boost bottom 30% magnitudes
        self.dims = dims

    def forward(self, x):
        z = fft.rfftn(x, dim=self.dims)  # real FFT for efficiency
        mag = torch.abs(z)

        # Compute threshold: e.g., 30th percentile of magnitudes
        thresh = torch.quantile(mag, self.thresh_quantile, dim=self.dims, keepdim=True)

        # Mask for underused (weak) freq: amplify them
        weak_mask = (mag < thresh).float()
        scale = 1.0 + (self.amp_factor - 1.0) * weak_mask  # boost by amp_factor where weak

        z_amp = z * scale
        y = fft.irfftn(z_amp, dim=self.dims)  # back to spatial
        return y

# Example residual block with freq amp on the skip (or on f(x) — test both)
class FreqAmpResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()  # or your fReLU
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Freq amp on the skip path
        self.freq_amp = FreqAmp(amp_factor=1.5, thresh_quantile=0.4)  # tune these

        # Projection if channels change
        self.proj = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        # Amp on identity (skip)
        if self.proj is not None:
            identity = self.proj(identity)
        identity = self.freq_amp(identity)  # or put this on 'out' instead

        out = out + identity
        out = self.relu(out)  # post-add activation
        return out

# To use: Stack these in a model, e.g., replace your sequential blocks
class FreqAmpNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            FreqAmpResBlock(3, 64),
            nn.MaxPool2d(2),
            FreqAmpResBlock(64, 128),
            nn.MaxPool2d(2),
            FreqAmpResBlock(128, 256),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

In [ ]:
class FreqAmp(nn.Module):
    """Amplifies underused freq components (low-magnitude ones)."""
    def __init__(self, amp_factor=2.0, thresh_quantile=0.3, dims=(-2, -1)):
        super().__init__()
        self.amp_factor = amp_factor  # how much to boost weak freq
        self.thresh_quantile = thresh_quantile  # e.g., boost bottom 30% magnitudes
        self.dims = dims

    def forward(self, x):
        z = fft.rfftn(x, dim=self.dims)  # real FFT for efficiency
        mag = torch.abs(z)

        # Compute threshold: e.g., 30th percentile of magnitudes
        # To compute quantile over multiple dimensions (H, W), flatten them first.
        original_mag_shape = mag.shape # (B, C, H, W_fft_out)
        mag_flat_spatial = mag.view(original_mag_shape[0], original_mag_shape[1], -1)

        # Calculate quantile along the flattened spatial dimension
        thresh = torch.quantile(mag_flat_spatial, self.thresh_quantile, dim=-1, keepdim=True)

        # Reshape thresh to (B, C, 1, 1) for proper broadcasting with original mag shape (B, C, H, W_fft_out)
        thresh = thresh.unsqueeze(-1) # Makes it (B, C, 1, 1)

        # Mask for underused (weak) freq: amplify them
        weak_mask = (mag < thresh).float()
        scale = 1.0 + (self.amp_factor - 1.0) * weak_mask  # boost by amp_factor where weak

        z_amp = z * scale
        # Use the original spatial dimensions of x for irfftn to ensure correct output size
        y = fft.irfftn(z_amp, dim=self.dims, s=x.shape[self.dims[0]:])
        return y

# Example residual block with freq amp on the skip (or on f(x) — test both)
class FreqAmpResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()  # or your fReLU
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Freq amp on the skip path
        self.freq_amp = FreqAmp(amp_factor=1.5, thresh_quantile=0.4)  # tune these

        # Projection if channels change
        self.proj = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        # Amp on identity (skip)
        if self.proj is not None:
            identity = self.proj(identity)
        identity = self.freq_amp(identity)  # or put this on 'out' instead

        out = out + identity
        out = self.relu(out)  # post-add activation
        return out

# To use: Stack these in a model, e.g., replace your sequential blocks
class FreqAmpNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            FreqAmpResBlock(3, 64),
            nn.MaxPool2d(2),
            FreqAmpResBlock(64, 128),
            nn.MaxPool2d(2),
            FreqAmpResBlock(128, 256),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

model = FreqAmpNet()
train(model, epochs=40)
torch.save(model.state_dict(), "./data/basic-architecture-with-freq-amp-res-blocks-40-epochs")

Starting training...
[1,   100] loss: 1.793 | acc: 38.82%
[1,   200] loss: 1.513 | acc: 53.84%
[1,   300] loss: 1.370 | acc: 60.89%
Epoch  1  ─  Test accuracy: 59.55%
[2,   100] loss: 1.225 | acc: 68.74%
[2,   200] loss: 1.184 | acc: 70.80%
[2,   300] loss: 1.130 | acc: 73.57%
Epoch  2  ─  Test accuracy: 70.04%
[3,   100] loss: 1.051 | acc: 77.47%
[3,   200] loss: 1.030 | acc: 77.97%
[3,   300] loss: 1.024 | acc: 78.47%
Epoch  3  ─  Test accuracy: 72.04%
[4,   100] loss: 0.974 | acc: 80.34%
[4,   200] loss: 0.955 | acc: 81.95%
[4,   300] loss: 0.942 | acc: 82.19%
Epoch  4  ─  Test accuracy: 79.69%
[5,   100] loss: 0.907 | acc: 84.26%
[5,   200] loss: 0.907 | acc: 83.84%
[5,   300] loss: 0.905 | acc: 83.95%
Epoch  5  ─  Test accuracy: 79.00%
[6,   100] loss: 0.870 | acc: 85.61%
[6,   200] loss: 0.867 | acc: 86.05%
[6,   300] loss: 0.869 | acc: 85.55%
Epoch  6  ─  Test accuracy: 81.83%
[7,   100] loss: 0.838 | acc: 87.10%
[7,   200] loss: 0.837 | acc: 87.18%
[7,   300] loss: 0.846 | acc: